# Cookie 导入流程测试

测试三个步骤：
1. **从浏览器读取** SESSDATA + bili_jct
2. **写入 .env 文件**
3. **验证** — 调用 B站 API 获取用户信息

## 准备工作：设置项目路径

In [ ]:
import sys
from pathlib import Path

# 确保项目根目录在 sys.path 中
project_root = Path.cwd()  # notebook 放在项目根目录
if str(project_root) not in sys.path:
    sys.path.insert(0, str(project_root))

print(f"项目根目录: {project_root}")
print(f"Python: {sys.version}")

## 第一步：从浏览器读取 Cookie

In [ ]:
from src.cookie_importer import import_cookies

try:
    cookies = import_cookies(browser="auto")
    print("✅ 成功读取 Cookie！")
    print(f"   SESSDATA 长度: {len(cookies.sessdata)}")
    print(f"   SESSDATA 前20字符: {cookies.sessdata[:20]}...")
    print(f"   bili_jct 长度: {len(cookies.bili_jct)}")
    print(f"   bili_jct: {cookies.bili_jct}")
except Exception as e:
    print(f"❌ 读取失败: {e}")

## 第二步：写入 .env 文件

In [ ]:
from src.cookie_importer import save_to_env

env_path = save_to_env(cookies)
print(f"✅ .env 已保存到: {env_path}")

# 读取确认
print("\n--- .env 文件内容 ---")
for line in env_path.read_text(encoding="utf-8").splitlines():
    if line.strip():
        # 隐藏敏感值
        if "=" in line and not line.startswith("#"):
            key, _, _ = line.partition("=")
            if key.strip() in ("BAF_SESSDATA", "BAF_BILI_JCT"):
                print(f"{key}=***已隐藏***")
                continue
        print(line)

# 加载配置确认
from src.config import Config
config = Config.from_env()
print(f"\n--- 配置加载结果 ---")
print(f"  auth_mode:  {config.auth_mode}")
print(f"  sessdata:   {'***已设置***' if config.sessdata else 'None'}")
print(f"  bili_jct:   {'***已设置***' if config.bili_jct else 'None'}")

## 第三步：验证 — 调用 B站 API 获取当前用户信息

In [ ]:
from bilibili_api import user, Credential

if not config.sessdata or not config.bili_jct:
    print("⚠️ 没有 Cookie，跳过验证。请先执行第一步。")
else:
    cred = Credential(sessdata=config.sessdata, bili_jct=config.bili_jct)
    print("正在调用 B站 API: user.get_self_info() …")
    
    try:
        info = await user.get_self_info(credential=cred)
        print("✅ 验证成功！Cookie 有效。")
        print(f"   用户名 (name):  {info.get('name')}")
        print(f"   用户 ID (mid):  {info.get('mid')}")
        print(f"   性别 (sex):     {info.get('sex')}")
        print(f"   等级 (level):   {info.get('level')}")
        print(f"   头像 (face):    {info.get('face')}")
        print(f"   签名 (sign):    {info.get('sign')}")
        print(f"   硬币 (coins):   {info.get('coins')}")
        
        # 显示完整返回字段
        print(f"\n--- 完整返回字段 ---")
        for k, v in info.items():
            val_str = str(v)
            if len(val_str) > 80:
                val_str = val_str[:80] + "..."
            print(f"  {k}: {val_str}")
            
    except Exception as e:
        print(f"❌ 验证失败: {e}")
        print(f"   → Cookie 可能已过期或无效")

## 补充：单独测试 DPAPI 解密（Chrome/Edge）

In [ ]:
# 这一步可以独立运行，不依赖前面的单元格
from src.cookie_importer import _chrome_cookie_path, _edge_cookie_path, _read_chrome_edge

for label, path_fn in [("Chrome", _chrome_cookie_path), ("Edge", _edge_cookie_path)]:
    db_path = path_fn()
    if db_path is None:
        print(f"❌ {label}: 未找到 Cookie 数据库")
        continue
    print(f"📁 {label} 数据库路径: {db_path}")
    print(f"   文件大小: {db_path.stat().st_size:,} bytes")
    
    try:
        cookies = _read_chrome_edge(db_path)
        bilibili_keys = [k for k in cookies if k in ("SESSDATA", "bili_jct", "DedeUserID", "bili_ticket")]
        print(f"   B站 相关 Cookie 数量: {len(bilibili_keys)}")
        for k in bilibili_keys:
            v = cookies[k]
            print(f"     {k}: {'✅ 有值' if v else '⚠️ 空值'} (长度: {len(v)})")
        # 显示所有 key
        all_keys = sorted(cookies.keys())
        print(f"   全部 Cookie key ({len(all_keys)} 个): {all_keys[:10]}{'...' if len(all_keys) > 10 else ''}")
    except Exception as e:
        print(f"   ❌ 解密失败: {e}")

## 补充：单独测试 Firefox Cookie 读取

In [ ]:
from src.cookie_importer import _firefox_cookie_path, _read_firefox

db_path = _firefox_cookie_path()
if db_path is None:
    print("❌ 未找到 Firefox Cookie 数据库")
else:
    print(f"📁 Firefox 数据库路径: {db_path}")
    print(f"   文件大小: {db_path.stat().st_size:,} bytes")
    
    try:
        cookies = _read_firefox(db_path)
        bilibili_keys = [k for k in cookies if k in ("SESSDATA", "bili_jct")]
        print(f"   B站 相关 Cookie: {bilibili_keys}")
        for k in bilibili_keys:
            v = cookies[k]
            print(f"     {k}: {'✅ 有值' if v else '⚠️ 空值'} (长度: {len(v)})")
    except Exception as e:
        print(f"   ❌ 读取失败: {e}")

## 补充：手动输入 Cookie 测试（如果自动读取失败）

In [ ]:
# 如果浏览器自动读取失败，可以手动设置 Cookie 进行测试
from src.cookie_importer import CookiePair, save_to_env
from bilibili_api import user, Credential

# 修改以下值为你自己的 Cookie
MANUAL_SESSDATA = ""  # 填入你的 SESSDATA
MANUAL_BILI_JCT = ""  # 填入你的 bili_jct

if not MANUAL_SESSDATA or not MANUAL_BILI_JCT:
    print("ℹ️ 未填写手动 Cookie，跳过。如需测试请修改上方变量。")
else:
    cookies = CookiePair(sessdata=MANUAL_SESSDATA, bili_jct=MANUAL_BILI_JCT)
    env_path = save_to_env(cookies)
    print(f"✅ 已写入 .env: {env_path}")
    
    # 验证
    cred = Credential(sessdata=MANUAL_SESSDATA, bili_jct=MANUAL_BILI_JCT)
    try:
        info = await user.get_self_info(credential=cred)
        print(f"✅ 验证成功: {info.get('name')} (UID: {info.get('mid')})")
    except Exception as e:
        print(f"❌ 验证失败: {e}")